In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Đọc dữ liệu Parquet
path = "data/traffic/OpenStack_Parquet/CIDDS-001-internal-week1.parquet"
df = pd.read_parquet(path)

# 2. Chọn 10 tính năng và nhãn mục tiêu [cite: 331, 430]
features = ['Src IP', 'Src Port', 'Dest IP', 'Dest Port', 'Proto', 
            'Flags', 'ToS', 'Duration', 'Bytes', 'Packets']
X = df[features].copy()
y = df['Attack Type'] # Sử dụng nhãn Attack Type [cite: 331]

# 3. Tiền xử lý: Chuyển đổi dữ liệu dạng chữ sang số (Label Encoding) 
le_dict = {}
for col in ['Src IP', 'Dest IP', 'Proto', 'Flags']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# 4. Chia tập Train/Test theo tỷ lệ 70/30 (Holdout) [cite: 433]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 5. Chuẩn hóa dữ liệu về khoảng 0-1 (Tránh Data Leakage) 
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Huấn luyện Random Forest với thông số từ bài báo 
rf_model = RandomForestClassifier(
    n_estimators=10, 
    criterion='gini',
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt', # 'auto' trong sklearn tương đương 'sqrt'
    class_weight='balanced',
    random_state=42
)

print("Đang huấn luyện mô hình...")
rf_model.fit(X_train_scaled, y_train)

# 7. Dự đoán và đánh giá [cite: 467]
y_pred = rf_model.predict(X_test_scaled)
print("\nKết quả đánh giá trên tập Test (Tuần 1):")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

KeyError: "['Src IP', 'Src Port', 'Dest IP', 'Dest Port', 'ToS'] not in index"

In [4]:
print(*df.columns, sep="\n")


Date first seen
Duration
Proto
Src IP Addr
Src Pt
Dst IP Addr
Dst Pt
Packets
Bytes
Flows
Flags
Tos
class
attackType
attackID
attackDescription


In [5]:
df.head()

,Date first seen,Duration,Proto,Src IP Addr,Src Pt,Dst IP Addr,Dst Pt,Packets,Bytes,Flows,Flags,Tos,class,attackType,attackID,attackDescription
0,2017-03-15 00:01:16.632,0.000,TCP,192.168.100.5,445,192.168.220.16,58844,1,108,1,.AP...,0,normal,---,---,---
1,2017-03-15 00:01:16.552,0.000,TCP,192.168.100.5,445,192.168.220.15,48888,1,108,1,.AP...,0,normal,---,---,---
2,2017-03-15 00:01:16.551,0.004,TCP,192.168.220.15,48888,192.168.100.5,445,2,174,1,.AP...,0,normal,---,---,---
3,2017-03-15 00:01:16.631,0.004,TCP,192.168.220.16,58844,192.168.100.5,445,2,174,1,.AP...,0,normal,---,---,---
4,2017-03-15 00:01:16.552,0.000,TCP,192.168.100.5,445,192.168.220.15,48888,1,108,1,.AP...,0,normal,---,---,---


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Đọc dữ liệu
df = pd.read_parquet("data/traffic/OpenStack_Parquet/CIDDS-001-internal-week1.parquet")

# HÀM XỬ LÝ K/M TRONG BYTES/PACKETS (Theo bài báo )
def clean_numeric(x):
    if isinstance(x, str):
        x = x.upper()
        if 'M' in x: return float(x.replace('M', '')) * 1000000
        if 'K' in x: return float(x.replace('K', '')) * 1000
    try:
        return float(x)
    except:
        return 0.0

# Làm sạch dữ liệu số
df['Bytes'] = df['Bytes'].apply(clean_numeric)
df['Packets'] = df['Packets'].apply(clean_numeric)

# 2. Chọn 10 tính năng dựa trên danh sách cột của bạn [cite: 430]
# Lưu ý: 'Tos' trong file của bạn viết là chữ 'o' thường
features = [
    'Src IP Addr', 'Src Pt', 'Dst IP Addr', 'Dst Pt', 'Proto', 
    'Flags', 'Tos', 'Duration', 'Bytes', 'Packets'
]

X = df[features].copy()
# Nhãn mục tiêu là 'attackType' [cite: 331]
# Thay thế giá trị mặc định '---' thành 'No Attack'
y = df['attackType'].replace("---", "No Attack")

# 3. Encoding các cột dạng chữ sang số
le_dict = {}
categorical_cols = ['Src IP Addr', 'Dst IP Addr', 'Proto', 'Flags']
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# 4. Chia tập dữ liệu 70/30 (Holdout method [cite: 433])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 5. Chuẩn hóa về [0, 1] [cite: 430]
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Random Forest với thông số chuẩn từ bài báo 
rf_model = RandomForestClassifier(
    n_estimators=10,            # Số lượng cây 
    criterion='gini',           # Tiêu chí Gini 
    min_samples_split=2,        # 
    min_samples_leaf=1,         # 
    max_features=None,          # Sử dụng tất cả tính năng 
    class_weight='balanced',    # Xử lý mất cân bằng lớp 
    random_state=42
)

print("Đang huấn luyện mô hình Random Forest...")
rf_model.fit(X_train_scaled, y_train)

# 7. Đánh giá kết quả [cite: 467]
y_pred = rf_model.predict(X_test_scaled)
print("\nBáo cáo kết quả phân loại (Tuần 1):")
print(classification_report(y_test, y_pred))

Đang huấn luyện mô hình Random Forest...

Báo cáo kết quả phân loại (Tuần 1):
              precision    recall  f1-score   support

   No Attack       1.00      1.00      1.00   2103269
  bruteForce       0.37      0.98      0.54       488
         dos       1.00      1.00      1.00    375638
    pingScan       0.40      0.82      0.54      1008
    portScan       1.00      0.97      0.98     55053

    accuracy                           1.00   2535456
   macro avg       0.75      0.96      0.81   2535456
weighted avg       1.00      1.00      1.00   2535456



In [8]:
df.head()

,Date first seen,Duration,Proto,Src IP Addr,Src Pt,Dst IP Addr,Dst Pt,Packets,Bytes,Flows,Flags,Tos,class,attackType,attackID,attackDescription
0,2017-03-15 00:01:16.632,0.000,TCP,192.168.100.5,445,192.168.220.16,58844,1.0,108.0,1,.AP...,0,normal,---,---,---
1,2017-03-15 00:01:16.552,0.000,TCP,192.168.100.5,445,192.168.220.15,48888,1.0,108.0,1,.AP...,0,normal,---,---,---
2,2017-03-15 00:01:16.551,0.004,TCP,192.168.220.15,48888,192.168.100.5,445,2.0,174.0,1,.AP...,0,normal,---,---,---
3,2017-03-15 00:01:16.631,0.004,TCP,192.168.220.16,58844,192.168.100.5,445,2.0,174.0,1,.AP...,0,normal,---,---,---
4,2017-03-15 00:01:16.552,0.000,TCP,192.168.100.5,445,192.168.220.15,48888,1.0,108.0,1,.AP...,0,normal,---,---,---


In [10]:
for x in df["attackType"].unique() : print(x) 

---
portScan
dos
pingScan
bruteForce
